In [237]:
from datetime import datetime
import sqlite3
import os
import uuid
import subprocess
import json
from pathlib import Path

print("Libraries imported successfully")

Libraries imported successfully


# Get Directories

Todo:
- [ ] Figure out better connection open/close logic 

In [238]:
def get_all_directories(db_path='boilest.db'):
    """Return all rows from the `directories` table as a list of (guid, path)."""
    conn = None
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute("SELECT guid, path FROM directories")
        rows = cur.fetchall()
        conn.close()
        return rows
    except Exception as e:
        print(f"✗ Error reading directories table: {e}")
        try:
            if conn:
                conn.close()
        except:
            pass
        return []

In [239]:
print(get_all_directories(db_path='boilest.db'))

[('44fb6f99-dd47-434e-9b3a-5e44c3f1fc48', '/Media'), ('1ae5df46-7d84-44ec-8174-73f1d12170c5', '/Anime'), ('eb120fd2-5860-409d-9c47-534cefd0dc9b', '/TV'), ('1a2c2141-cc92-43a2-8c41-9ce95f0864dd', '/Movies')]


# Directory Scanner

In [240]:
# Note: `write_file_to_database` removed from this cell. Use central database utilities instead.

def scan_directories_and_enqueue(directory_path, directory_guid=None, extensions=None):
    if extensions is None:
        extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']
    directory_path = os.path.expanduser(directory_path)
    if not os.path.isdir(directory_path):
        print(f'Directory not found: {directory_path}')
        return
    for root, dirs, files in os.walk(directory_path):
        for file in files:
            for ext in extensions:
                if file.lower().endswith(ext.lower()):
                    file_path = os.path.join(root, file)
                    yield {
                        'directory_guid': directory_guid,
                        'root': root,
                        'file': file,
                        'file_path': file_path
                    }
                    break  # Only match one extension per file


## Test Function

In [241]:
def print_scanned_files(directory_path, directory_guid=None, extensions=None):
    """Call `scan_directories_and_enqueue` and pretty-print each yielded item."""
    count = 0
    for item in scan_directories_and_enqueue(directory_path, directory_guid, extensions):
        try:
            print(json.dumps(item, indent=2))
        except Exception:
            print(item)
        count += 1
    print(f"\nTotal files yielded: {count}")


print_scanned_files("/media")
# Example: call with current working directory

{
  "directory_guid": null,
  "root": "/media\\Media 1",
  "file": "test_file_01.mp4",
  "file_path": "/media\\Media 1\\test_file_01.mp4"
}
{
  "directory_guid": null,
  "root": "/media\\Media 1\\Media A",
  "file": "test_file_02.mp4",
  "file_path": "/media\\Media 1\\Media A\\test_file_02.mp4"
}
{
  "directory_guid": null,
  "root": "/media\\Media 1\\Media A",
  "file": "test_file_03.mp4",
  "file_path": "/media\\Media 1\\Media A\\test_file_03.mp4"
}
{
  "directory_guid": null,
  "root": "/media\\Media 2",
  "file": "test_file_04.mp4",
  "file_path": "/media\\Media 2\\test_file_04.mp4"
}
{
  "directory_guid": null,
  "root": "/media\\Media 2",
  "file": "test_file_05.mp4",
  "file_path": "/media\\Media 2\\test_file_05.mp4"
}
{
  "directory_guid": null,
  "root": "/media\\Media 3",
  "file": "test_file_06.mp4",
  "file_path": "/media\\Media 3\\test_file_06.mp4"
}
{
  "directory_guid": null,
  "root": "/media\\Media 4",
  "file": "test.mkv",
  "file_path": "/media\\Media 4\\test.mkv"
}


# FFProbe Function

Todo:
- [ ] Research expanding the entries on the ffprobe string for HDR and other criteria

In [242]:
def run_ffprobe(file_path):
    try:
        cmd = [
            'ffprobe',
            '-loglevel', 'quiet',
            '-show_entries', 'format:stream=index,stream,codec_type,codec_name,channel_layout,format=nb_streams',  
            '-of', 'json',
            file_path
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode != 0:
            return {'error': f'ffprobe failed: {result.stderr}'}
        
        probe_data = json.loads(result.stdout)
        
        # Pretty-print the ffprobe JSON output
        print(json.dumps(probe_data, indent=2))
        
        return probe_data
    
    except FileNotFoundError:
        return {'error': 'ffprobe not found. Ensure ffmpeg is installed and in PATH.'}
    except subprocess.TimeoutExpired:
        return {'error': 'ffprobe timeout (file too large or network issue)'}
    except json.JSONDecodeError:
        return {'error': 'Invalid ffprobe JSON output'}
    except Exception as e:
        return {'error': str(e)}


In [243]:
file_path = '/media\\Media 4\\test_file_07.MP4'
print(run_ffprobe(file_path))

{
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "h264",
      "codec_type": "video"
    }
  ],
  "format": {
    "filename": "/media\\Media 4\\test_file_07.MP4",
    "nb_streams": 1,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "mov,mp4,m4a,3gp,3g2,mj2",
    "format_long_name": "QuickTime / MOV",
    "start_time": "0.000000",
    "duration": "4.254250",
    "size": "775034",
    "bit_rate": "1457430",
    "probe_score": 100,
    "tags": {
      "major_brand": "isom",
      "minor_version": "512",
      "compatible_brands": "isomiso2avc1mp41",
      "encoder": "Lavf58.76.100"
    }
  }
}
{'programs': [], 'stream_groups': [], 'streams': [{'index': 0, 'codec_name': 'h264', 'codec_type': 'video'}], 'format': {'filename': '/media\\Media 4\\test_file_07.MP4', 'nb_streams': 1, 'nb_programs': 0, 'nb_stream_groups': 0, 'format_name': 'mov,mp4,m4a,3gp,3g2,mj2', 'format_long_name': 'QuickTime / MOV', 'start_time': '0.000

# Check Codecs
Loops through the streams in stream_info from requires_encoding, then calls functions to determine if the steam needs encoding based on stream type conditions 

Todo:
- [x] Copy over the stream looping function from Boilest v1.0
- [ ] Research SVT-AV1 best practices for various media types
- [ ] Store SVT-AV1 best practice presets in the DB
- [ ] Call best-practive presets in check_video_stream
- [ ] Determine what audio codec to go with
- [ ] Determine what the compromises will be if ASS subtitles are re-encoded as SubRip
- [ ] Determine if there are consequences for deleting attachments 

In [244]:
def check_codecs(encoding_decision,stream_info, ffmpeg_command):
    streams_count = stream_info['format']['nb_streams']
    
    for i in range (0,streams_count):
        codec_type = stream_info['streams'][i]['codec_type'] 
        if codec_type == 'video':
            print('Stream ' + str(i) + ' is video')
            encoding_decision, ffmpeg_command = check_video_stream(encoding_decision, i, stream_info, ffmpeg_command)
        elif codec_type == 'audio':
            encoding_decision, ffmpeg_command = check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command)
            print('audio stream')
        elif codec_type == 'subtitle':
            encoding_decision, ffmpeg_command = check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command)
            print('subtitle stream')
        elif codec_type == 'attachment':
            encoding_decision, ffmpeg_command = check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command) 
            print('attachment stream')    
    print (encoding_decision)   
    print (ffmpeg_command)
    return encoding_decision, ffmpeg_command

def check_video_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the video stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    desired_video_codec = 'av1'
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    if codec_name == desired_video_codec:
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name == 'mjpeg':
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v copy'
    elif codec_name != desired_video_codec: 
        encoding_decision = True
        svt_av1_string = "libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15"
        ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:v ' + svt_av1_string
    else:
        print('ignoring for now')
    return encoding_decision, ffmpeg_command


def check_audio_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the audio stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_audio_codec = 'aac'
    #if codec_name != desired_video_codec:
    #    encoding_decision = True
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:a copy'
    return encoding_decision, ffmpeg_command


def check_subtitle_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the subtitle stream from check_codecs to determine if the stream needs encoding
    codec_name = stream_info['streams'][i]['codec_name'] 
    # This will be populated at a later date
    #desired_subtitle_codec = 'srt'
    #if codec_name != desired_subtitle_codec:
    #    encoding_decision = True
    print('Steam ' + str(i) + ' codec is: ' + codec_name)
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:s copy'
    return encoding_decision, ffmpeg_command


def check_attachmeent_stream(encoding_decision, i, stream_info, ffmpeg_command):
    # Checks the attachment stream from check_codecs to determine if the stream needs encoding
    # This will be populated at a later date
    #desired_attachment_codec = '???'
    #if codec_name != desired_attachment_codec:
    #    encoding_decision = True
    # Note, attachments may not have a codec name if the attachment is an image
    ffmpeg_command = ffmpeg_command + ' -map 0:' + str(i) + ' -c:t copy'
    return encoding_decision, ffmpeg_command

In [245]:

file_path = '/media\\Media 4\\test_file_07.MP4'
encoding_decision = 'False'
stream_info = run_ffprobe(file_path)
ffmpeg_command = ''
print(check_codecs(encoding_decision,stream_info, ffmpeg_command))

{
  "programs": [],
  "stream_groups": [],
  "streams": [
    {
      "index": 0,
      "codec_name": "h264",
      "codec_type": "video"
    }
  ],
  "format": {
    "filename": "/media\\Media 4\\test_file_07.MP4",
    "nb_streams": 1,
    "nb_programs": 0,
    "nb_stream_groups": 0,
    "format_name": "mov,mp4,m4a,3gp,3g2,mj2",
    "format_long_name": "QuickTime / MOV",
    "start_time": "0.000000",
    "duration": "4.254250",
    "size": "775034",
    "bit_rate": "1457430",
    "probe_score": 100,
    "tags": {
      "major_brand": "isom",
      "minor_version": "512",
      "compatible_brands": "isomiso2avc1mp41",
      "encoder": "Lavf58.76.100"
    }
  }
}
Stream 0 is video
Steam 0 codec is: h264
True
 -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise=0:tune=0:enable-qm=1:qm-min=0:qm-max=15
(True, ' -map 0:0 -c:v libsvtav1 -crf 25 -preset 4 -g 240 -pix_fmt yuv420p10le -svtav1-params filmgrain=20:film-grain-denoise

# File Output Naming Function

In [246]:
import os

def output_file_name(file_path, encoding_decision):
    # Get the filename and current extension
    filename = os.path.basename(file_path)
    name_without_ext = os.path.splitext(filename)[0]
    current_ext = os.path.splitext(filename)[1]
    
    # Change extension to .mkv if it's not already
    if current_ext.lower() != '.mkv':
        new_filename = name_without_ext + '.mkv'
        encoding_decision = True
    else:
        new_filename = filename
    
    # Return just the new filename without any directory path
    return new_filename, encoding_decision

In [247]:
file_path = '/media\\Media 4\\test_file_07.MP4'
encoding_decision = False
print(output_file_name(file_path, encoding_decision))


('test_file_07.mkv', True)


# Get File Size Function

In [248]:
def get_file_size_kb(file_path):
    try:
        file_size_bytes = Path(file_path).stat().st_size
        file_size_kb = int(file_size_bytes / 1024)
        return file_size_kb
    except FileNotFoundError:
        print(f"✗ File not found: {file_path}")
        return 0
    except Exception as e:
        print(f"✗ Error getting file size: {e}")
        return 0

In [249]:
file_path = '/media\\Media 4\\test_file_07.MP4'
print(get_file_size_kb(file_path))

756


# Write to Queue Function

Todo:

In [250]:
def write_to_queue(file_path, output_file_name, ffmpeg_string, file_size, directory_guid):
    """Write an entry into the `queue` table in boilest.db."""
    try:
        guid = str(uuid.uuid4())
        date_added = datetime.now().isoformat()
        input_file_name = os.path.basename(file_path)

        db_path = 'boilest.db'
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()

        # Resolve directory path from provided directory GUID if possible
        directory_path = None
        try:
            cur.execute("SELECT path FROM directories WHERE guid = ?", (directory_guid,))
            row = cur.fetchone()
            if row:
                directory_path = row[0]
        except Exception:
            directory_path = None

        # Fallback to dirname of file_path if directory_path not found
        if not directory_path:
            directory_path = os.path.dirname(file_path)

        cur.execute("INSERT INTO queue (guid, directory_path, input_file_name, output_file_name, before_file_size, ffmpeg_string, date_added) VALUES (?, ?, ?, ?, ?, ?, ?)",
                    (guid, directory_path, input_file_name, output_file_name, file_size, ffmpeg_string, date_added))
        conn.commit()
        conn.close()
        print(f"✓ Wrote queue entry {guid} for {input_file_name}")
        return guid
    except Exception as e:
        print(f"✗ Error writing to queue: {e}")
        try:
            conn.close()
        except:
            pass
        return None


In [251]:
file_path = '/stuff/things'
output_file_name = 'test_output.mkv'
ffmpeg_string = 'things343'
file_size = 423432
directory_guid = 'stuffweqwe'

print(write_to_queue(file_path, output_file_name, ffmpeg_string, file_size, directory_guid))

✓ Wrote queue entry 576875e3-6457-4ed8-86f3-e5e53424cd17 for things
576875e3-6457-4ed8-86f3-e5e53424cd17


# Pulling it all together

In [ ]:


db_path='boilest.db'

#directoriess = get_all_directories(db_path)
# normalize to list of dicts for printing
#print('Directories (get_all_directories):')
#print(json.dumps(directoriess, indent=2))

file_path = '/media\\Media 4\\test_file_07.MP4'


def write_to_db (file_path):
    encoding_decision = False
    file_size = get_file_size_kb(file_path)
    new_filename, encoding_decision = output_file_name(file_path, encoding_decision)
    stream_info = run_ffprobe(file_path)
    ffmpeg_command = ''
    # check_codecs returns (encoding_decision, ffmpeg_command)
    encoding_decision, ffmpeg_command = check_codecs(encoding_decision, stream_info, ffmpeg_command)

    print (file_path)
    print (file_size)
    print (new_filename)
    print (ffmpeg_command)
    print(bool(encoding_decision))

write_to_db(file_path)


SyntaxError: Missing parentheses in call to 'print'. Did you mean print(...)? (2056444791.py, line 24)